# Notebook 안내
원본 팀 연구 코드의 계산 로직은 변경하지 않았다. 가독성을 위한 설명 셀만 추가했으며 개인정보·데이터 노출 방지를 위해 기존 실행 출력은 제거했다. 원천 데이터는 포함하지 않으며 독립 재실행은 하지 않았다. README의 결과 및 한계 참고.


# PAS × Export Rank 50:50 — Fixed 30 + Full Robustness

`PAS_Export_Rank50_VolatilityManaged(1).ipynb`의 **PAS 및 Export 결합식은 유지**하고,
메인 포트폴리오를 **Long 30 / Short 30 고정**으로 변경한 연구용 버전입니다.

포함 분석:
- PAS Only vs PAS + Export Rank 50:50
- Long30 / Short30, Gross 200%
- 거래비용 / 대차비용 / execution filter
- 전체기간 및 subperiod 성과
- **Q1~Q5 quintile test**
- Q1~Q5 누적수익 그래프 + Q5−Q1 NAV
- **Q5−Q1 Newey-West(HAC) t-stat / p-value**
- 연도별 quintile 수익률 및 Q5−Q1
- **5×5 hotspot: Market-cap quintile × PAS+Export score quintile**
- 5×5 평균 월수익률 / annualized return / 평균 종목 수 heatmap
- 각 size quintile 내부 Q5−Q1 spread


### 코드 설명 — 데이터 로딩과 정렬: DataGuide 시장·산업 패널, 수출 CSV를 결합하고 수출 YoY 변화에 1개월 lag를 적용한다. DART 수집 코드는 포함되지 않는다.


In [ ]:
%matplotlib inline
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

START = "2015-01-01"
AUM = 1e11
FRAC = 0.05
FEE = 0.002
BORROW = 0.03
PART = 0.10
DAYS = 5
MIN_EXEC = 0.80

# VM: 최대 Gross200, 고변동성에서만 축소
VAR_FLOOR_Q = 0.05
CAP_TARGET_Q = 0.50
CAP_MAX_SCALE = 1.0

DATA_DIR = Path("Data")
if not (DATA_DIR / "adj_close_0906.csv").exists():
    DATA_DIR = Path(".") if Path("adj_close_0906.csv").exists() else Path("..") / "Data"

price = pd.read_csv(DATA_DIR / "adj_close_0906.csv", index_col="Date", parse_dates=True).apply(pd.to_numeric, errors="coerce")
total = pd.read_csv(DATA_DIR / "adj_close_dividend_0906.csv", index_col="Date", parse_dates=True).apply(pd.to_numeric, errors="coerce")
cap = pd.read_csv(DATA_DIR / "mkt_cap_0906.csv", index_col="Date", parse_dates=True).apply(pd.to_numeric, errors="coerce")
tv = pd.read_csv(DATA_DIR / "20일평균거래대금.csv", index_col=0, parse_dates=True)
ind = pd.read_csv(DATA_DIR / "Industry27.csv", index_col="Date", parse_dates=True)
export = pd.read_csv(DATA_DIR / "export_monthly.csv", index_col=0, parse_dates=True)

tv = tv.apply(lambda s: pd.to_numeric(s.astype("string").str.replace(",", "", regex=False), errors="coerce"))

stocks = price.columns.intersection(total.columns).intersection(cap.columns).intersection(ind.columns)
price, total, cap, ind = price[stocks], total[stocks], cap[stocks], ind[stocks]
tv = tv.reindex(index=price.index, columns=stocks).where(lambda x: x >= 0)

# 월말 + PAS
ret = price.pct_change(fill_method=None)
dates = pd.DatetimeIndex(price.groupby(price.index.to_period("M")).apply(lambda x: x.index[-1]).values)
pm = price.reindex(dates)

mom = pm.shift(1) / pm.shift(12) - 1
vol = ret.rolling(126, min_periods=100).std().reindex(dates)
skew = ret.rolling(126, min_periods=100).skew().reindex(dates)

cap_m = cap.reindex(cap.index.union(dates)).sort_index().ffill().reindex(dates)
ind_m = ind.reindex(ind.index.union(dates)).sort_index().ffill().reindex(dates)
total_m = total.reindex(total.index.union(dates)).sort_index().ffill().reindex(dates)
next_ret = total_m.shift(-1) / total_m - 1

# 업로드한 PAS X Export(2)와 동일한 Export Shock
yoy_cols = {
    "반도체":"Semiconductor_YoY",
    "자동차":"Automobile_YoY",
    "자동차부품":"Auto_Parts_YoY",
    "조선":"Ship_YoY",
    "화학":"Petrochemical_YoY",
    "디스플레이":"Display_YoY"
}

shock = pd.DataFrame(index=export.index)
for k,v in yoy_cols.items():
    if v in export.columns:
        shock[k] = pd.to_numeric(export[v], errors="coerce") - pd.to_numeric(export[v], errors="coerce").shift(1)

# look-ahead 방지: 1개월 lag
shock = shock.shift(1)

def export_group(x):
    x = str(x)
    if "반도체" in x: return "반도체"
    if "자동차부품" in x or "자동차 부품" in x: return "자동차부품"
    if "자동차" in x: return "자동차"
    if "조선" in x: return "조선"
    if "화학" in x or "석유화학" in x: return "화학"
    if "디스플레이" in x: return "디스플레이"
    return np.nan

print("기간:", price.index.min(), "~", price.index.max())
print("종목 수:", len(stocks))
print("Export shock columns:", list(shock.columns))


### 코드 설명 — 가격 신호 및 산업별 수출 신호 결합: 순위를 계산하고 미매핑 종목은 가격 신호를 유지한다.


In [ ]:
# ============================================================
# 2) 공통 score 생성
#    기존 파일의 PAS + Export 50:50 결합식을 그대로 사용
# ============================================================

N_STOCKS = 30

def build_scores(t, use_export=True):
    c=cap_m.loc[t].dropna()
    univ=c[c.rank(pct=True)>=.70].index

    x=pd.DataFrame({
        "MOM":mom.loc[t],
        "VOL":vol.loc[t],
        "SKEW":skew.loc[t]
    }).dropna()
    x=x.loc[x.index.intersection(univ)].copy()
    if x.empty:
        return x

    r=x[["MOM","VOL","SKEW"]].rank(pct=True)
    x["LONG"]=r["MOM"]*(1-r["SKEW"])
    x["PAS"]=r["MOM"]*(1-r["VOL"])*(1-r["SKEW"])
    x["LONG_R"]=x["LONG"].rank(pct=True)
    x["PAS_R"]=x["PAS"].rank(pct=True)

    x["LONG_SCORE"]=x["LONG_R"]
    x["SHORT_SCORE"]=x["PAS_R"]
    x["EXPORT_R"]=np.nan

    if use_export:
        available=shock.index[shock.index<=t]
        if len(available):
            groups=ind_m.loc[t].reindex(x.index).map(export_group)
            er=groups.map(shock.loc[available[-1]]).rank(pct=True)
            mask=er.notna()
            x.loc[mask,"EXPORT_R"]=er.loc[mask]
            x.loc[mask,"LONG_SCORE"]=(x.loc[mask,"LONG_R"]+er.loc[mask])/2
            x.loc[mask,"SHORT_SCORE"]=(x.loc[mask,"PAS_R"]+er.loc[mask])/2

    # 연구용 단일 composite score:
    # Long/Short 비대칭 구조를 보존하면서 high=good이 되도록 평균
    x["COMPOSITE_SCORE"]=(x["LONG_SCORE"]+x["SHORT_SCORE"])/2
    return x

def resolve_next_returns(t, names):
    if t not in next_ret.index:
        return pd.Series(index=names,dtype=float)
    ri=next_ret.loc[t].reindex(names).copy()
    if ri.isna().any():
        pos=np.where(dates==t)[0]
        if len(pos) and pos[0]+1<len(dates):
            end=dates[pos[0]+1]
            missing=ri.index[ri.isna()]
            last=total.loc[t:end,missing].where(lambda z:z>0).ffill().iloc[-1]
            ri.loc[missing]=last/total.loc[t,missing]-1
    return ri


### 코드 설명 — 고정 종목 수 백테스트: 다음 달 수익률, 거래 비용, 대차 및 유동성 조건을 반영한다.


In [ ]:
# ============================================================
# 3) Fixed Long30 / Short30 백테스트
# ============================================================

def build_fixed30(use_export):
    LW=pd.DataFrame(0.0,index=dates,columns=stocks)
    SW=pd.DataFrame(0.0,index=dates,columns=stocks)
    diag=[]

    for t in dates[:-1]:
        if t<pd.Timestamp(START): continue
        x=build_scores(t,use_export)
        if x.empty: continue

        n=N_STOCKS
        adv=tv.loc[t].reindex(x.index)  # Input is already ADV20 (KRW/day).
        required_adv=(AUM/n)*MIN_EXEC/(PART*DAYS)

        eligible=x[
            adv.ge(required_adv)
            & price.loc[t].reindex(x.index).gt(0)
            & total.loc[t].reindex(x.index).gt(0)
        ].copy()

        if len(eligible)<2*n: continue

        shorts=eligible.nsmallest(n,"SHORT_SCORE").index
        longs=eligible.drop(index=shorts).nlargest(n,"LONG_SCORE").index
        if len(longs)!=n or len(shorts)!=n: continue

        LW.loc[t,longs]=1/n
        SW.loc[t,shorts]=-1/n
        diag.append({"Date":t,"Eligible":len(eligible),
                     "LongN":len(longs),"ShortN":len(shorts),
                     "ExportMapped":int(eligible["EXPORT_R"].notna().sum())})

    return LW,SW,pd.DataFrame(diag)

weights={}
for name,use_exp in {"PAS_Only":False,"PAS_Export_50_50":True}.items():
    weights[name]=build_fixed30(use_exp)

def backtest(LW,SW):
    rows=[]; old=pd.Series(dtype=float)
    for t,end in zip(dates[:-1],dates[1:]):
        if t<pd.Timestamp(START): continue
        w=(LW.loc[t]+SW.loc[t]); w=w[w!=0]
        if len(w)!=2*N_STOCKS: continue
        ri=resolve_next_returns(t,w.index)
        if ri.isna().any(): continue
        ix=w.index.union(old.index)
        turnover=(w.reindex(ix,fill_value=0)-old.reindex(ix,fill_value=0)).abs().sum()
        gross=float(w@ri)
        trade=FEE*turnover
        borrow=abs(w[w<0].sum())*BORROW/12
        rows.append({"FormationDate":t,"Date":end,"Gross":gross,
                     "Net":gross-trade-borrow,"Turnover":turnover,
                     "TradingCost":trade,"BorrowCost":borrow})
        old=w
    return pd.DataFrame(rows).set_index("Date")

bt={k:backtest(v[0],v[1]) for k,v in weights.items()}

def perf_stats(r):
    r=r.dropna()
    if len(r)==0:return pd.Series(dtype=float)
    nav=(1+r).cumprod(); years=len(r)/12
    cagr=nav.iloc[-1]**(1/years)-1 if nav.iloc[-1]>0 else np.nan
    vol=r.std(ddof=1)*np.sqrt(12)
    dd=nav/nav.cummax()-1
    return pd.Series({"Months":len(r),"MeanMonthly":r.mean(),"CAGR":cagr,
                      "Vol":vol,"Sharpe":r.mean()/r.std(ddof=1)*np.sqrt(12),
                      "MDD":dd.min()})

summary=pd.DataFrame({k:perf_stats(v["Net"]) for k,v in bt.items()}).T
print("=== FIXED 30 NET PERFORMANCE ===")
display(summary.round(4))

# subperiod
rows=[]
for k,v in bt.items():
    for p,(a,b) in {"2015-2021":("2015-01-01","2021-12-31"),
                    "2022-present":("2022-01-01","2099-12-31")}.items():
        z=perf_stats(v.loc[a:b,"Net"]);z["Strategy"]=k;z["Period"]=p;rows.append(z)
subperiod=pd.DataFrame(rows).set_index(["Strategy","Period"])
print("=== SUBPERIOD ===");display(subperiod.round(4))

plt.figure(figsize=(10,5))
for k,v in bt.items():
    ((1+v["Net"]).cumprod()).plot(label=k)
plt.title("Fixed30 PAS vs PAS + Export 50:50 — Net NAV")
plt.ylabel("NAV");plt.legend();plt.grid(alpha=.25);plt.tight_layout();plt.show()


### 코드 설명 — 분위 수익률 검증: 스코어 순위와 이후 수익률 관계를 확인한다.


In [ ]:
# ============================================================
# 4) Quintile test — PAS+Export composite score
#    시총 상위30% + execution-eligible universe에서 Q1~Q5
# ============================================================

qrows=[]
for t in dates[:-1]:
    if t<pd.Timestamp(START): continue
    x=build_scores(t,True)
    if x.empty: continue

    # 동일 execution 조건
    adv=tv.loc[t].reindex(x.index)  # Input is already ADV20 (KRW/day).
    required_adv=(AUM/N_STOCKS)*MIN_EXEC/(PART*DAYS)
    ok=adv.ge(required_adv) & price.loc[t].reindex(x.index).gt(0) & total.loc[t].reindex(x.index).gt(0)
    x=x.loc[ok].copy()
    if len(x)<25: continue

    rr=resolve_next_returns(t,x.index)
    z=pd.DataFrame({"Score":x["COMPOSITE_SCORE"],"Return":rr}).dropna()
    if len(z)<25: continue

    z["Q"]=pd.qcut(z["Score"].rank(method="first"),5,labels=[1,2,3,4,5])
    for q,g in z.groupby("Q",observed=True):
        qrows.append({"Date":t,"Q":int(q),"Return":g["Return"].mean(),"N":len(g)})

qpanel=pd.DataFrame(qrows)
qret=qpanel.pivot(index="Date",columns="Q",values="Return").rename(columns=lambda q:f"Q{q}")
qsum=pd.DataFrame({q:perf_stats(qret[q]) for q in qret.columns}).T

print("=== QUINTILE PERFORMANCE ===")
display(qsum.round(4))

# Q1~Q5 cumulative NAV
plt.figure(figsize=(10,5))
for q in qret.columns:
    ((1+qret[q].dropna()).cumprod()).plot(label=q)
plt.title("PAS + Export 50:50 — Q1 to Q5 cumulative NAV")
plt.ylabel("NAV");plt.legend(ncol=5);plt.grid(alpha=.25);plt.tight_layout();plt.show()

# Q5-Q1
spread=(qret["Q5"]-qret["Q1"]).dropna()
plt.figure(figsize=(10,4.5))
((1+spread).cumprod()).plot(label="Q5 - Q1")
plt.title("PAS + Export 50:50 — Q5-Q1 cumulative NAV")
plt.ylabel("NAV");plt.legend();plt.grid(alpha=.25);plt.tight_layout();plt.show()


### 코드 설명 — Newey–West HAC(6) 평균 차이 검정 및 기간별 분석.


In [ ]:
# ============================================================
# 5) Robustness — Newey-West, subperiod, annual quintiles
# ============================================================
import statsmodels.api as sm

def nw_mean_test(s,maxlags=6):
    s=s.dropna()
    if len(s)<12:return pd.Series({"Mean":np.nan,"NW_t":np.nan,"NW_p":np.nan,"N":len(s)})
    fit=sm.OLS(s.values,np.ones((len(s),1))).fit(cov_type="HAC",cov_kwds={"maxlags":maxlags})
    return pd.Series({"Mean":s.mean(),"NW_t":fit.tvalues[0],"NW_p":fit.pvalues[0],"N":len(s)})

nw_all=nw_mean_test(spread,6).to_frame("Full sample").T
print("=== Q5-Q1 NEWey-WEST (HAC, 6 lags) ===")
display(nw_all.round(5))

subrows=[]
for p,(a,b) in {"2015-2021":("2015-01-01","2021-12-31"),
                "2022-present":("2022-01-01","2099-12-31")}.items():
    z=spread.loc[a:b]
    st=perf_stats(z)
    nw=nw_mean_test(z,6)
    row=pd.concat([st,nw[["NW_t","NW_p"]]])
    row["Period"]=p;subrows.append(row)
q5q1_subperiod=pd.DataFrame(subrows).set_index("Period")
print("=== Q5-Q1 SUBPERIOD ===");display(q5q1_subperiod.round(4))

annual_q=qret.groupby(qret.index.year).mean()
annual_q["Q5-Q1"]=annual_q["Q5"]-annual_q["Q1"]
print("=== ANNUAL QUINTILE MEAN MONTHLY RETURNS ===")
display((annual_q*100).round(2))

plt.figure(figsize=(11,5))
for q in ["Q1","Q2","Q3","Q4","Q5"]:
    plt.plot(annual_q.index,annual_q[q]*100,marker="o",label=q)
plt.axhline(0,linewidth=.8)
plt.title("Annual average monthly return by quintile")
plt.ylabel("% per month");plt.xlabel("Year");plt.legend(ncol=5)
plt.grid(alpha=.25);plt.tight_layout();plt.show()

plt.figure(figsize=(10,4))
plt.bar(annual_q.index.astype(str),annual_q["Q5-Q1"]*100)
plt.axhline(0,linewidth=.8)
plt.title("Annual Q5-Q1 mean monthly spread")
plt.ylabel("% per month");plt.xticks(rotation=45)
plt.grid(axis="y",alpha=.25);plt.tight_layout();plt.show()


### 코드 설명 — 규모와 신호를 이중 정렬해 5×5 결과를 확인한다.


In [ ]:
# ============================================================
# 6) 5×5 HOTSPOT
#    Market-cap quintile × PAS+Export composite-score quintile
#    전체 시총 상위30% + execution universe 안에서 독립적으로 5분위
# ============================================================

rows=[]
for t in dates[:-1]:
    if t<pd.Timestamp(START): continue
    x=build_scores(t,True)
    if x.empty:continue

    adv=tv.loc[t].reindex(x.index)  # Input is already ADV20 (KRW/day).
    required_adv=(AUM/N_STOCKS)*MIN_EXEC/(PART*DAYS)
    ok=adv.ge(required_adv) & price.loc[t].reindex(x.index).gt(0) & total.loc[t].reindex(x.index).gt(0)
    x=x.loc[ok].copy()
    if len(x)<50:continue

    rr=resolve_next_returns(t,x.index)
    z=pd.DataFrame({
        "Cap":cap_m.loc[t].reindex(x.index),
        "Score":x["COMPOSITE_SCORE"],
        "Return":rr
    }).dropna()
    if len(z)<50:continue

    z["C_Q"]=pd.qcut(z["Cap"].rank(method="first"),5,labels=[1,2,3,4,5])
    z["S_Q"]=pd.qcut(z["Score"].rank(method="first"),5,labels=[1,2,3,4,5])

    for (cq,sq),g in z.groupby(["C_Q","S_Q"],observed=True):
        rows.append({"Date":t,"C_Q":int(cq),"S_Q":int(sq),
                     "Return":g["Return"].mean(),"N":len(g)})

double_sort=pd.DataFrame(rows)

mean5=double_sort.pivot_table(index="C_Q",columns="S_Q",values="Return",aggfunc="mean").reindex(index=range(1,6),columns=range(1,6))
n5=double_sort.pivot_table(index="C_Q",columns="S_Q",values="N",aggfunc="mean").reindex(index=range(1,6),columns=range(1,6))
ann5=(1+mean5)**12-1

mean5.index=[f"C{i}" for i in range(1,6)];mean5.columns=[f"Q{i}" for i in range(1,6)]
n5.index=mean5.index;n5.columns=mean5.columns
ann5.index=mean5.index;ann5.columns=mean5.columns

print("=== 5×5 Mean Monthly Return ===");display((mean5*100).round(2))
print("=== 5×5 Annualized Return ===");display((ann5*100).round(2))
print("=== 5×5 Average N ===");display(n5.round(1))

def heatmap(df,title,percent=True):
    fig,ax=plt.subplots(figsize=(8,6))
    vals=df.values*100 if percent else df.values
    im=ax.imshow(vals,aspect="auto")
    ax.set_xticks(range(5),df.columns)
    ax.set_yticks(range(5),df.index)
    ax.set_xlabel("PAS + Export score quintile (Q1 low → Q5 high)")
    ax.set_ylabel("Market-cap quintile (C1 small → C5 large)")
    ax.set_title(title)
    fig.colorbar(im,ax=ax)
    for i in range(5):
        for j in range(5):
            v=vals[i,j]
            if np.isfinite(v):
                ax.text(j,i,f"{v:.2f}%" if percent else f"{v:.1f}",
                        ha="center",va="center",fontsize=9)
    plt.tight_layout();plt.show()

heatmap(mean5,"5×5 Hotspot — Mean monthly return",True)
heatmap(ann5,"5×5 Hotspot — Annualized return",True)
heatmap(n5,"5×5 Hotspot — Average stock count",False)

within_size=pd.DataFrame({
    "Q5-Q1 MeanMonthly":mean5["Q5"]-mean5["Q1"]
})
within_size["Q5-Q1 Annualized"]=(1+within_size["Q5-Q1 MeanMonthly"])**12-1
print("=== Q5-Q1 WITHIN EACH SIZE QUINTILE ===")
display(within_size.round(4))

plt.figure(figsize=(8,4))
plt.bar(within_size.index,within_size["Q5-Q1 MeanMonthly"]*100)
plt.axhline(0,linewidth=.8)
plt.title("Q5-Q1 spread within each market-cap quintile")
plt.ylabel("Mean monthly spread (%)");plt.grid(axis="y",alpha=.25)
plt.tight_layout();plt.show()


### 코드 설명 — 수출 결합 비중 변경 및 결과 저장.


In [ ]:
# ============================================================
# 7) 추가 강건성 — 결합 가중치 민감도
#    Export weight 0%, 25%, 50%, 75%, 100%
# ============================================================

weight_rows=[]
for ew in [0.00,0.25,0.50,0.75,1.00]:
    monthly=[]
    for t in dates[:-1]:
        if t<pd.Timestamp(START):continue
        x=build_scores(t,True)
        if x.empty:continue

        # mapped 종목은 weight sensitivity, unmapped는 PAS 그대로
        long_s=x["LONG_R"].copy()
        short_s=x["PAS_R"].copy()
        m=x["EXPORT_R"].notna()
        long_s.loc[m]=(1-ew)*x.loc[m,"LONG_R"]+ew*x.loc[m,"EXPORT_R"]
        short_s.loc[m]=(1-ew)*x.loc[m,"PAS_R"]+ew*x.loc[m,"EXPORT_R"]

        adv=tv.loc[t].reindex(x.index)  # Input is already ADV20 (KRW/day).
        req=(AUM/N_STOCKS)*MIN_EXEC/(PART*DAYS)
        ok=adv.ge(req)&price.loc[t].reindex(x.index).gt(0)&total.loc[t].reindex(x.index).gt(0)
        tmp=pd.DataFrame({"L":long_s,"S":short_s}).loc[ok].dropna()
        if len(tmp)<60:continue

        shorts=tmp.nsmallest(N_STOCKS,"S").index
        longs=tmp.drop(index=shorts).nlargest(N_STOCKS,"L").index
        rr=resolve_next_returns(t,longs.union(shorts))
        if rr.isna().any():continue
        monthly.append((t,rr.reindex(longs).mean()-rr.reindex(shorts).mean()))

    s=pd.Series(dict(monthly)).sort_index()
    st=perf_stats(s)
    st["ExportWeight"]=ew
    weight_rows.append(st)

weight_sensitivity=pd.DataFrame(weight_rows).set_index("ExportWeight")
print("=== EXPORT WEIGHT SENSITIVITY (gross signal return) ===")
display(weight_sensitivity.round(4))

plt.figure(figsize=(8,4))
plt.plot(weight_sensitivity.index*100,weight_sensitivity["Sharpe"],marker="o")
plt.title("Export-weight sensitivity — Sharpe")
plt.xlabel("Export weight (%)");plt.ylabel("Sharpe")
plt.grid(alpha=.25);plt.tight_layout();plt.show()

# SAVE
summary.to_csv("fixed30_pas_export_summary.csv",encoding="utf-8-sig")
subperiod.to_csv("fixed30_pas_export_subperiod.csv",encoding="utf-8-sig")
qsum.to_csv("fixed30_pas_export_quintile_summary.csv",encoding="utf-8-sig")
qret.to_csv("fixed30_pas_export_quintile_returns.csv",encoding="utf-8-sig")
nw_all.to_csv("fixed30_pas_export_newey_west.csv",encoding="utf-8-sig")
q5q1_subperiod.to_csv("fixed30_pas_export_q5q1_subperiod.csv",encoding="utf-8-sig")
annual_q.to_csv("fixed30_pas_export_annual_quintiles.csv",encoding="utf-8-sig")
mean5.to_csv("fixed30_pas_export_5x5_mean_monthly.csv",encoding="utf-8-sig")
ann5.to_csv("fixed30_pas_export_5x5_annualized.csv",encoding="utf-8-sig")
n5.to_csv("fixed30_pas_export_5x5_avg_n.csv",encoding="utf-8-sig")
within_size.to_csv("fixed30_pas_export_5x5_within_size.csv",encoding="utf-8-sig")
weight_sensitivity.to_csv("fixed30_pas_export_weight_sensitivity.csv",encoding="utf-8-sig")

print("모든 분석 완료.")


# 추가 검증: 실제 매매신호와 동일한 Quintile + Net Export-weight Sensitivity

아래 셀은 기존 결과를 유지하면서 두 가지를 수정/검증합니다.

1. **실제 Long/Short 매매 신호를 각각 그대로 사용한 quintile test**
   - Long side: 실제 `LONG_SCORE`
   - Short side: 실제 `SHORT_SCORE`
   - 따라서 임의의 `COMPOSITE_SCORE` 평균에 의존하지 않음
   - Long은 Q5−Q1, Short는 **Q1−Q5**가 실제 포트폴리오 방향과 대응

2. **Export weight sensitivity를 메인 백테스트와 완전히 같은 Net 기준으로 재계산**
   - Long30 / Short30
   - 실제 turnover
   - 거래비용 20bp one-way
   - 대차비용 연 3%
   - 0%, 25%, 50%, 75%, 100%를 동일 조건에서 비교


### 코드 설명 — 롱·숏 실제 신호별 분위 분석.


In [ ]:
# ============================================================
# A. 실제 Long / Short signal별 quintile
# ============================================================

def signal_quintile_test(signal_col):
    rows=[]
    for t in dates[:-1]:
        if t<pd.Timestamp(START): continue
        x=build_scores(t,True)
        if x.empty: continue

        adv=tv.loc[t].reindex(x.index)  # Input is already ADV20 (KRW/day).
        req=(AUM/N_STOCKS)*MIN_EXEC/(PART*DAYS)
        ok=adv.ge(req) & price.loc[t].reindex(x.index).gt(0) & total.loc[t].reindex(x.index).gt(0)
        x=x.loc[ok].copy()
        if len(x)<25: continue

        rr=resolve_next_returns(t,x.index)
        z=pd.DataFrame({"Score":x[signal_col],"Return":rr}).dropna()
        if len(z)<25: continue

        z["Q"]=pd.qcut(z["Score"].rank(method="first"),5,labels=[1,2,3,4,5])
        for q,g in z.groupby("Q",observed=True):
            rows.append({"Date":t,"Q":int(q),"Return":g["Return"].mean(),"N":len(g)})

    panel=pd.DataFrame(rows)
    ret=panel.pivot(index="Date",columns="Q",values="Return").rename(columns=lambda q:f"Q{q}")
    summ=pd.DataFrame({q:perf_stats(ret[q]) for q in ret.columns}).T
    return panel,ret,summ

long_qpanel,long_qret,long_qsum=signal_quintile_test("LONG_SCORE")
short_qpanel,short_qret,short_qsum=signal_quintile_test("SHORT_SCORE")

print("=== ACTUAL LONG_SCORE QUINTILES ===")
display(long_qsum.round(4))
print("Long direction spread: Q5 - Q1")
long_spread=(long_qret["Q5"]-long_qret["Q1"]).dropna()
display(pd.concat([perf_stats(long_spread),nw_mean_test(long_spread,6)[["NW_t","NW_p"]]]).to_frame("Long Q5-Q1").T.round(5))

print("\n=== ACTUAL SHORT_SCORE QUINTILES ===")
display(short_qsum.round(4))
print("Short portfolio direction spread: Q1 - Q5")
short_spread=(short_qret["Q1"]-short_qret["Q5"]).dropna()
display(pd.concat([perf_stats(short_spread),nw_mean_test(short_spread,6)[["NW_t","NW_p"]]]).to_frame("Short Q1-Q5").T.round(5))

# 그래프
plt.figure(figsize=(10,5))
for q in long_qret.columns:
    ((1+long_qret[q].dropna()).cumprod()).plot(label=q)
plt.title("Actual LONG_SCORE — Q1 to Q5 cumulative NAV")
plt.ylabel("NAV");plt.legend(ncol=5);plt.grid(alpha=.25);plt.tight_layout();plt.show()

plt.figure(figsize=(10,5))
for q in short_qret.columns:
    ((1+short_qret[q].dropna()).cumprod()).plot(label=q)
plt.title("Actual SHORT_SCORE — Q1 to Q5 cumulative NAV")
plt.ylabel("NAV");plt.legend(ncol=5);plt.grid(alpha=.25);plt.tight_layout();plt.show()

plt.figure(figsize=(10,4.5))
((1+long_spread).cumprod()).plot(label="Long signal: Q5-Q1")
((1+short_spread).cumprod()).plot(label="Short signal: Q1-Q5")
plt.title("Actual trading-signal quintile spreads")
plt.ylabel("NAV");plt.legend();plt.grid(alpha=.25);plt.tight_layout();plt.show()


### 코드 설명 — 동일 비용 조건의 수출 비중 민감도.


In [ ]:
# ============================================================
# B. Export weight sensitivity — SAME NET BACKTEST as main test
# ============================================================

def build_weighted_fixed30(export_weight):
    LW=pd.DataFrame(0.0,index=dates,columns=stocks)
    SW=pd.DataFrame(0.0,index=dates,columns=stocks)

    for t in dates[:-1]:
        if t<pd.Timestamp(START): continue

        # Export information까지 포함해 기본 구성요소 확보
        x=build_scores(t,True)
        if x.empty: continue

        # unmapped는 PAS 그대로, mapped만 지정한 weight로 결합
        L=x["LONG_R"].copy()
        S=x["PAS_R"].copy()
        m=x["EXPORT_R"].notna()
        L.loc[m]=(1-export_weight)*x.loc[m,"LONG_R"] + export_weight*x.loc[m,"EXPORT_R"]
        S.loc[m]=(1-export_weight)*x.loc[m,"PAS_R"] + export_weight*x.loc[m,"EXPORT_R"]

        adv=tv.loc[t].reindex(x.index)  # Input is already ADV20 (KRW/day).
        req=(AUM/N_STOCKS)*MIN_EXEC/(PART*DAYS)
        ok=adv.ge(req) & price.loc[t].reindex(x.index).gt(0) & total.loc[t].reindex(x.index).gt(0)

        tmp=pd.DataFrame({"LONG_SCORE":L,"SHORT_SCORE":S}).loc[ok].dropna()
        if len(tmp)<2*N_STOCKS: continue

        shorts=tmp.nsmallest(N_STOCKS,"SHORT_SCORE").index
        longs=tmp.drop(index=shorts).nlargest(N_STOCKS,"LONG_SCORE").index
        if len(longs)!=N_STOCKS or len(shorts)!=N_STOCKS: continue

        LW.loc[t,longs]=1/N_STOCKS
        SW.loc[t,shorts]=-1/N_STOCKS

    return LW,SW

net_sensitivity={}
sens_rows=[]

for ew in [0.00,0.25,0.50,0.75,1.00]:
    lw,sw=build_weighted_fixed30(ew)
    res=backtest(lw,sw)   # 메인 전략과 동일한 turnover/fee/borrow
    net_sensitivity[ew]=res

    st=perf_stats(res["Net"])
    st["ExportWeight"]=ew
    st["AvgTurnover"]=res["Turnover"].mean()
    st["AvgTradingCost"]=res["TradingCost"].mean()
    st["AvgBorrowCost"]=res["BorrowCost"].mean()
    sens_rows.append(st)

net_weight_sensitivity=pd.DataFrame(sens_rows).set_index("ExportWeight")

print("=== NET EXPORT-WEIGHT SENSITIVITY ===")
display(net_weight_sensitivity.round(5))

# CAGR / Sharpe / MDD를 각각 확인
fig,ax=plt.subplots(figsize=(8,4))
ax.plot(net_weight_sensitivity.index*100,net_weight_sensitivity["Sharpe"],marker="o")
ax.set_title("Net Export-weight sensitivity — Sharpe")
ax.set_xlabel("Export weight (%)");ax.set_ylabel("Net Sharpe")
ax.grid(alpha=.25);plt.tight_layout();plt.show()

fig,ax=plt.subplots(figsize=(8,4))
ax.plot(net_weight_sensitivity.index*100,net_weight_sensitivity["CAGR"]*100,marker="o")
ax.set_title("Net Export-weight sensitivity — CAGR")
ax.set_xlabel("Export weight (%)");ax.set_ylabel("Net CAGR (%)")
ax.grid(alpha=.25);plt.tight_layout();plt.show()

fig,ax=plt.subplots(figsize=(8,4))
ax.plot(net_weight_sensitivity.index*100,net_weight_sensitivity["MDD"]*100,marker="o")
ax.set_title("Net Export-weight sensitivity — MDD")
ax.set_xlabel("Export weight (%)");ax.set_ylabel("MDD (%)")
ax.grid(alpha=.25);plt.tight_layout();plt.show()

# 각 weight의 Net NAV
plt.figure(figsize=(10,5))
for ew,res in net_sensitivity.items():
    ((1+res["Net"]).cumprod()).plot(label=f"Export {int(ew*100)}%")
plt.title("Net NAV by Export weight — Long30 / Short30")
plt.ylabel("NAV");plt.legend();plt.grid(alpha=.25);plt.tight_layout();plt.show()


### 코드 설명 — 기간별 민감도와 결과 일치 여부 검토.


In [ ]:
# ============================================================
# C. 기간별 Net sensitivity + 기존 50:50과 일치 여부 확인
# ============================================================

sens_sub=[]
for ew,res in net_sensitivity.items():
    for pname,(a,b) in {
        "2015-2021":("2015-01-01","2021-12-31"),
        "2022-present":("2022-01-01","2099-12-31")
    }.items():
        z=perf_stats(res.loc[a:b,"Net"])
        z["ExportWeight"]=ew
        z["Period"]=pname
        sens_sub.append(z)

net_weight_subperiod=pd.DataFrame(sens_sub).set_index(["ExportWeight","Period"])
print("=== NET WEIGHT SENSITIVITY BY SUBPERIOD ===")
display(net_weight_subperiod.round(4))

# 0%는 PAS Only, 50%는 PAS+Export와 같은 구성인지 수익률로 확인
checks=[]
for ew,main_name in [(0.0,"PAS_Only"),(0.5,"PAS_Export_50_50")]:
    a=net_sensitivity[ew]["Net"]
    b=bt[main_name]["Net"]
    common=a.index.intersection(b.index)
    diff=(a.loc[common]-b.loc[common]).abs()
    checks.append({
        "Weight":ew,
        "MainStrategy":main_name,
        "CommonMonths":len(common),
        "MaxAbsNetReturnDiff":diff.max() if len(diff) else np.nan,
        "MeanAbsNetReturnDiff":diff.mean() if len(diff) else np.nan
    })

consistency_check=pd.DataFrame(checks)
print("=== CONSISTENCY CHECK ===")
display(consistency_check)

# 저장
long_qsum.to_csv("actual_long_signal_quintiles.csv",encoding="utf-8-sig")
short_qsum.to_csv("actual_short_signal_quintiles.csv",encoding="utf-8-sig")
long_qret.to_csv("actual_long_signal_quintile_returns.csv",encoding="utf-8-sig")
short_qret.to_csv("actual_short_signal_quintile_returns.csv",encoding="utf-8-sig")
net_weight_sensitivity.to_csv("net_export_weight_sensitivity.csv",encoding="utf-8-sig")
net_weight_subperiod.to_csv("net_export_weight_subperiod.csv",encoding="utf-8-sig")
consistency_check.to_csv("net_export_weight_consistency_check.csv",index=False,encoding="utf-8-sig")

print("추가 검증 완료.")


## 실제 Long / Short / Combined 회귀와 2,000억 원 체결 용량
각 1,000억 원, 하루 ADV20의 10%, 5일 누적 체결률 80% 기준. 회귀는 비용 차감 실제 포지션 수익률에 HAC(6개월)를 적용합니다. 시장 통제 회귀는 무위험수익률을 차감하지 않은 회귀입니다. 합산 수익률은 기존 1,000억 원 기준이며 총 익스포저 2,000억 원 기준 수익률도 별도 저장합니다.


### 코드 설명 — 별도 모듈로 체결량 및 시장 통제 회귀 진단. 시장 통제 절편과 초과수익 알파를 구분한다.


In [ ]:
from fixed30_execution_regression import run_diagnostics
EXECUTION_BASIS = "daily_participation_10pct"
verified_diagnostics = run_diagnostics(globals())


### 코드 설명 — 최근 선정 종목 확인.


In [ ]:
# ============================================================
# D. 최근 3개월 Long / Short 편입 종목 확인
# ============================================================

RECENT_SIGNAL_MONTHS = 3
RECENT_SIGNAL_STRATEGIES = {"PAS_Only": False, "PAS_Export_50_50": True}
RECENT_SIGNAL_FILE = "recent_3m_long_short_signals.csv"

def save_signal_csv(frame, filename):
    try:
        frame.to_csv(filename, index=False, encoding="utf-8-sig")
        return filename
    except PermissionError:
        fallback = Path(filename).with_name(
            f"{Path(filename).stem}_updated{Path(filename).suffix}"
        )
        frame.to_csv(fallback, index=False, encoding="utf-8-sig")
        print(f"기존 파일이 열려 있어 대체 파일에 저장: {fallback}")
        return str(fallback)

def collect_recent_signals(strategy, use_export, months=3, start=None, end=None):
    # 마지막 월말도 포함합니다. 아직 다음 달 수익률이 없어도 편입 종목은 확인할 수 있습니다.
    if start is None:
        formation_dates = pd.DatetimeIndex([
            t for t in dates if t >= pd.Timestamp(START)
        ])[-months:]
    else:
        formation_dates = pd.DatetimeIndex([
            t for t in dates
            if pd.Timestamp(start) <= t <= pd.Timestamp(end)
        ])
    rows = []

    for t in formation_dates:
        x = build_scores(t, use_export)
        if x.empty:
            continue

        adv = tv.loc[t].reindex(x.index)
        required_adv = (AUM / N_STOCKS) * MIN_EXEC / (PART * DAYS)
        eligible = x[
            adv.ge(required_adv)
            & price.loc[t].reindex(x.index).gt(0)
            & total.loc[t].reindex(x.index).gt(0)
        ].copy()
        if len(eligible) < 2 * N_STOCKS:
            continue

        shorts = eligible.nsmallest(N_STOCKS, "SHORT_SCORE").index
        longs = eligible.drop(index=shorts).nlargest(N_STOCKS, "LONG_SCORE").index
        if len(longs) != N_STOCKS or len(shorts) != N_STOCKS:
            continue

        for side, names, score_col in [
            ("Long", longs, "LONG_SCORE"),
            ("Short", shorts, "SHORT_SCORE"),
        ]:
            for rank, ticker in enumerate(names, start=1):
                rows.append({
                    "Strategy": strategy,
                    "FormationDate": t,
                    "Side": side,
                    "Rank": rank,
                    "Ticker": ticker,
                    "SignalScore": x.loc[ticker, score_col],
                    "LONG_SCORE": x.loc[ticker, "LONG_SCORE"],
                    "SHORT_SCORE": x.loc[ticker, "SHORT_SCORE"],
                    "EXPORT_R": x.loc[ticker, "EXPORT_R"],
                    "EligibleN": len(eligible),
                })

    return pd.DataFrame(rows)

recent_signal_frames = [
    collect_recent_signals(strategy, use_export, RECENT_SIGNAL_MONTHS)
    for strategy, use_export in RECENT_SIGNAL_STRATEGIES.items()
]
recent_signals = pd.concat(
    [frame for frame in recent_signal_frames if not frame.empty],
    ignore_index=True,
) if any(not frame.empty for frame in recent_signal_frames) else pd.DataFrame()

if recent_signals.empty:
    print("최근 3개월에 유효한 Long/Short 편입 종목이 없습니다.")
else:
    recent_signals = recent_signals.sort_values(
        ["Strategy", "FormationDate", "Side", "Rank"]
    )
    recent_signal_summary = (
        recent_signals.assign(Ticker=recent_signals["Ticker"].astype(str))
        .groupby(["Strategy", "FormationDate", "Side"])["Ticker"]
        .agg(", ".join)
        .unstack("Side")
        .reindex(columns=["Long", "Short"])
    )
    print("=== RECENT 3-MONTH LONG / SHORT SIGNALS ===")
    display(recent_signal_summary)
    recent_signal_saved_file = save_signal_csv(recent_signals, RECENT_SIGNAL_FILE)

# ============================================================
# E. 2022년 / 2023년 / 2026년 4~6월 월별 Long / Short 편입 종목
# ============================================================

HISTORICAL_SIGNAL_PERIODS = {
    "2022": ("2022-01-01", "2022-12-31"),
    "2023": ("2023-01-01", "2023-12-31"),
    "2026-04": ("2026-04-01", "2026-04-30"),
    "2026-05": ("2026-05-01", "2026-05-31"),
    "2026-06": ("2026-06-01", "2026-06-30"),
}
HISTORICAL_SIGNAL_FILE = "historical_long_short_signals.csv"

historical_signal_frames = []
for period, (period_start, period_end) in HISTORICAL_SIGNAL_PERIODS.items():
    for strategy, use_export in RECENT_SIGNAL_STRATEGIES.items():
        frame = collect_recent_signals(
            strategy, use_export, start=period_start, end=period_end
        )
        if not frame.empty:
            frame.insert(0, "Period", period)
            historical_signal_frames.append(frame)

historical_signals = (
    pd.concat(historical_signal_frames, ignore_index=True)
    if historical_signal_frames else pd.DataFrame()
)

if historical_signals.empty:
    print("2022년/2023년에 유효한 Long/Short 편입 종목이 없습니다.")
else:
    historical_signals = historical_signals.sort_values(
        ["Period", "Strategy", "FormationDate", "Side", "Rank"]
    )
    historical_signal_summary = (
        historical_signals.assign(Ticker=historical_signals["Ticker"].astype(str))
        .groupby(["Period", "Strategy", "FormationDate", "Side"])["Ticker"]
        .agg(", ".join)
        .unstack("Side")
        .reindex(columns=["Long", "Short"])
    )
    print("=== HISTORICAL MONTHLY LONG / SHORT SIGNALS ===")
    with pd.option_context("display.max_colwidth", None):
        display(historical_signal_summary)
    historical_signal_saved_file = save_signal_csv(
        historical_signals, HISTORICAL_SIGNAL_FILE
    )
    print(f"상세 결과 저장: {historical_signal_saved_file}")

if not recent_signals.empty:
    print(f"상세 결과 저장: {recent_signal_saved_file}")


### 코드 설명 — 요인 CSV를 이용해 무위험수익률 조정 성과를 계산한다. KOSPI200 자료가 없으면 KOSPI proxy를 표시한다.


In [ ]:
# ============================================================
# F. RF-adjusted robust Sharpe + KOSPI200 NAV comparison
# ============================================================

import statsmodels.api as sm

FACTOR_FILE = DATA_DIR / "Factor_0917.csv"
RF_ADJUSTED_SUMMARY_FILE = "rf_adjusted_sharpe_benchmark_summary.csv"
NAV_COMPARISON_FILE = "strategy_vs_kospi200_nav.csv"
KOSPI200_CANDIDATES = [
    DATA_DIR / "KOSPI200.csv",
    DATA_DIR / "kospi200.csv",
    Path("KOSPI200.csv"),
    Path("kospi200.csv"),
]

factor = (
    pd.read_csv(FACTOR_FILE, parse_dates=["Date"])
    .set_index("Date")
    .sort_index()
)
factor["Rf"] = pd.to_numeric(factor["Rf"], errors="coerce")
factor["KOSPI"] = pd.to_numeric(factor["KOSPI"], errors="coerce")

def load_benchmark_price():
    seen = set()
    for path in KOSPI200_CANDIDATES:
        path = Path(path)
        if str(path) in seen or not path.exists():
            continue
        seen.add(str(path))
        raw = pd.read_csv(path)
        date_col = next(
            (c for c in ["Date", "date", "DATE"] if c in raw.columns),
            raw.columns[0],
        )
        raw[date_col] = pd.to_datetime(raw[date_col], errors="coerce")
        value_col = next(
            (c for c in ["Adj Close", "Close", "Price", "KOSPI200", "Index", "Value"] if c in raw.columns),
            None,
        )
        if value_col is None:
            numeric = raw.drop(columns=[date_col]).apply(pd.to_numeric, errors="coerce")
            value_col = numeric.notna().sum().idxmax()
            raw[value_col] = numeric[value_col]
        series = pd.Series(
            pd.to_numeric(raw[value_col], errors="coerce").values,
            index=raw[date_col],
        ).dropna().sort_index()
        series = series[~series.index.duplicated(keep="last")]
        if not series.empty:
            return series, "KOSPI200", str(path)

    # Factor_0917에는 KOSPI200이 아니라 KOSPI만 있으므로 명시적으로 proxy로 표시합니다.
    return factor["KOSPI"].dropna(), "KOSPI_proxy", "Factor_0917.csv:KOSPI"

benchmark_price, benchmark_name, benchmark_source = load_benchmark_price()
if benchmark_name == "KOSPI_proxy":
    print("주의: KOSPI200 파일이 없어 Factor_0917의 KOSPI를 proxy로 사용합니다.")
    print("정확한 KOSPI200 비교를 하려면 KOSPI200.csv를 같은 폴더에 추가한 뒤 셀을 다시 실행하세요.")
else:
    print(f"KOSPI200 benchmark source: {benchmark_source}")

# Rf는 Factor_0917의 연율(%)이므로 월별 무위험수익률로 변환합니다.
rf_annual = factor["Rf"].dropna() / 100.0
rf_monthly = (
    (1.0 + rf_annual.groupby(rf_annual.index.to_period("M")).last()) ** (1.0 / 12.0) - 1.0
)
benchmark_monthly_level = benchmark_price.groupby(benchmark_price.index.to_period("M")).last()
benchmark_monthly_return = benchmark_monthly_level.pct_change()

def map_monthly(series, index):
    return pd.Series(
        [series.get(t.to_period("M"), np.nan) for t in index],
        index=index,
        dtype=float,
    )

def robust_rf_stats(returns, rf):
    sample = pd.concat([returns.rename("Return"), rf.rename("RF")], axis=1)
    sample = sample.replace([np.inf, -np.inf], np.nan).dropna()
    if len(sample) < 12:
        return {}
    excess = sample["Return"] - sample["RF"]
    fit = sm.OLS(excess.values, np.ones((len(excess), 1))).fit(
        cov_type="HAC", cov_kwds={"maxlags": 6}
    )
    nav = (1.0 + sample["Return"]).cumprod()
    years = len(sample) / 12.0
    drawdown = nav / nav.cummax() - 1.0
    vol = excess.std(ddof=1) * np.sqrt(12.0)
    return {
        "N": len(sample),
        "MeanMonthly": sample["Return"].mean(),
        "MeanExcessMonthly": excess.mean(),
        "RFAdjustedVolAnnual": vol,
        "RFAdjustedSharpe": excess.mean() / excess.std(ddof=1) * np.sqrt(12.0),
        "NW_t_Excess": fit.tvalues[0],
        "NW_p_Excess": fit.pvalues[0],
        "CAGR": nav.iloc[-1] ** (1.0 / years) - 1.0 if nav.iloc[-1] > 0 else np.nan,
        "MDD": drawdown.min(),
        "FinalNAV": nav.iloc[-1],
    }

summary_rows = []
nav_frames = []
for strategy, result in bt.items():
    strategy_return = result["Net"].copy()
    benchmark_return = map_monthly(benchmark_monthly_return, strategy_return.index)
    rf_return = map_monthly(rf_monthly, strategy_return.index)
    sample = pd.DataFrame({
        "StrategyReturn": strategy_return,
        "BenchmarkReturn": benchmark_return,
        "RFMonthly": rf_return,
    }).replace([np.inf, -np.inf], np.nan).dropna()
    if len(sample) < 12:
        continue

    sample["StrategyExcessReturn"] = sample["StrategyReturn"] - sample["RFMonthly"]
    sample["BenchmarkExcessReturn"] = sample["BenchmarkReturn"] - sample["RFMonthly"]
    sample["ActiveReturn"] = sample["StrategyReturn"] - sample["BenchmarkReturn"]
    sample["StrategyNAV"] = (1.0 + sample["StrategyReturn"]).cumprod()
    sample["BenchmarkNAV"] = (1.0 + sample["BenchmarkReturn"]).cumprod()
    sample["RelativeNAV"] = sample["StrategyNAV"] / sample["BenchmarkNAV"]

    strategy_stats = robust_rf_stats(sample["StrategyReturn"], sample["RFMonthly"])
    benchmark_stats = robust_rf_stats(sample["BenchmarkReturn"], sample["RFMonthly"])
    active_vol = sample["ActiveReturn"].std(ddof=1) * np.sqrt(12.0)
    summary_rows.append({
        "Strategy": strategy,
        "Benchmark": benchmark_name,
        "BenchmarkSource": benchmark_source,
        "N": len(sample),
        "StrategyRFAdjustedSharpe": strategy_stats["RFAdjustedSharpe"],
        "BenchmarkRFAdjustedSharpe": benchmark_stats["RFAdjustedSharpe"],
        "StrategyNW_t_Excess": strategy_stats["NW_t_Excess"],
        "StrategyNW_p_Excess": strategy_stats["NW_p_Excess"],
        "StrategyCAGR": strategy_stats["CAGR"],
        "BenchmarkCAGR": benchmark_stats["CAGR"],
        "StrategyMDD": strategy_stats["MDD"],
        "BenchmarkMDD": benchmark_stats["MDD"],
        "StrategyFinalNAV": sample["StrategyNAV"].iloc[-1],
        "BenchmarkFinalNAV": sample["BenchmarkNAV"].iloc[-1],
        "ActiveMeanMonthly": sample["ActiveReturn"].mean(),
        "ActiveVolAnnual": active_vol,
        "ActiveSharpe": sample["ActiveReturn"].mean() / active_vol * np.sqrt(12.0) if active_vol > 0 else np.nan,
    })
    nav_frames.append(sample.assign(Strategy=strategy, Benchmark=benchmark_name, BenchmarkSource=benchmark_source))

rf_sharpe_benchmark_summary = pd.DataFrame(summary_rows).set_index("Strategy")
strategy_vs_benchmark_nav = pd.concat(nav_frames).reset_index(names="Date")
print("=== RF-ADJUSTED ROBUST SHARPE / BENCHMARK SUMMARY ===")
display(rf_sharpe_benchmark_summary.round(5))
print("=== NAV COMPARISON: final rows ===")
display(strategy_vs_benchmark_nav.groupby("Strategy").tail(1))

rf_sharpe_benchmark_summary.to_csv(RF_ADJUSTED_SUMMARY_FILE, encoding="utf-8-sig")
strategy_vs_benchmark_nav.to_csv(NAV_COMPARISON_FILE, index=False, encoding="utf-8-sig")
print(f"요약 CSV 저장: {RF_ADJUSTED_SUMMARY_FILE}")
print(f"월별 NAV CSV 저장: {NAV_COMPARISON_FILE}")

plt.figure(figsize=(11, 5))
for strategy, group in strategy_vs_benchmark_nav.groupby("Strategy"):
    group = group.sort_values("Date").set_index("Date")
    group["StrategyNAV"].plot(label=f"{strategy} NAV")
    if strategy == list(bt.keys())[0]:
        group["BenchmarkNAV"].plot(label=f"{benchmark_name} NAV", linestyle="--", color="black")
plt.title("Strategy NAV vs benchmark NAV")
plt.ylabel("NAV (common start = 1.0)")
plt.legend(); plt.grid(alpha=.25); plt.tight_layout(); plt.show()


## 2022년부터 최근까지 전략 NAV와 KOSPI200 벤치마크 비교

2022-01-01 이후 공통 월별 표본을 비교합니다. KOSPI200.csv가 있으면 실제 KOSPI200을 사용하고, 파일이 없으면 KOSPI proxy를 사용하며 CSV에 실제 출처를 기록합니다.


### 코드 설명 — 2022년 이후 공통 표본 비교.


In [ ]:
# ============================================================
# G. 2022-present NAV comparison and CSV export
# ============================================================
NAV_START_2022 = pd.Timestamp("2022-01-01")
NAV_2022_FILE = "strategy_vs_kospi200_nav_2022_present.csv"
NAV_2022_SUMMARY_FILE = "strategy_vs_kospi200_nav_2022_present_summary.csv"

nav_2022_frames = []
nav_2022_summary_rows = []

for strategy, result in bt.items():
    strategy_return = result["Net"].loc[result.index >= NAV_START_2022]
    benchmark_return = map_monthly(benchmark_monthly_return, strategy_return.index)
    sample = pd.DataFrame({
        "StrategyReturn": strategy_return,
        "BenchmarkReturn": benchmark_return,
    }).replace([np.inf, -np.inf], np.nan).dropna()

    if sample.empty:
        continue

    sample["ActiveReturn"] = sample["StrategyReturn"] - sample["BenchmarkReturn"]
    sample["StrategyNAV"] = (1.0 + sample["StrategyReturn"]).cumprod()
    sample["BenchmarkNAV"] = (1.0 + sample["BenchmarkReturn"]).cumprod()
    sample["RelativeNAV"] = sample["StrategyNAV"] / sample["BenchmarkNAV"]
    sample["Strategy"] = strategy
    sample["Benchmark"] = benchmark_name
    sample["BenchmarkSource"] = benchmark_source
    nav_2022_frames.append(sample)

    strategy_nav = sample["StrategyNAV"]
    benchmark_nav = sample["BenchmarkNAV"]
    strategy_dd = strategy_nav / strategy_nav.cummax() - 1.0
    benchmark_dd = benchmark_nav / benchmark_nav.cummax() - 1.0
    years = len(sample) / 12.0
    nav_2022_summary_rows.append({
        "Strategy": strategy,
        "Benchmark": benchmark_name,
        "BenchmarkSource": benchmark_source,
        "StartDate": sample.index.min(),
        "EndDate": sample.index.max(),
        "Months": len(sample),
        "StrategyCumulativeReturn": strategy_nav.iloc[-1] - 1.0,
        "BenchmarkCumulativeReturn": benchmark_nav.iloc[-1] - 1.0,
        "ActiveCumulativeReturn": sample["RelativeNAV"].iloc[-1] - 1.0,
        "StrategyCAGR": strategy_nav.iloc[-1] ** (1.0 / years) - 1.0 if strategy_nav.iloc[-1] > 0 else np.nan,
        "BenchmarkCAGR": benchmark_nav.iloc[-1] ** (1.0 / years) - 1.0 if benchmark_nav.iloc[-1] > 0 else np.nan,
        "StrategyMDD": strategy_dd.min(),
        "BenchmarkMDD": benchmark_dd.min(),
    })

if not nav_2022_frames:
    raise ValueError("2022년 이후 전략과 벤치마크가 겹치는 월별 표본이 없습니다.")

strategy_vs_kospi200_nav_2022 = pd.concat(nav_2022_frames).reset_index(names="Date")
strategy_vs_kospi200_nav_2022_summary = pd.DataFrame(nav_2022_summary_rows).set_index("Strategy")

strategy_vs_kospi200_nav_2022.to_csv(NAV_2022_FILE, index=False, encoding="utf-8-sig")
strategy_vs_kospi200_nav_2022_summary.to_csv(NAV_2022_SUMMARY_FILE, encoding="utf-8-sig")

print(f"2022-present benchmark: {benchmark_name} ({benchmark_source})")
print(f"월별 NAV CSV 저장: {NAV_2022_FILE}")
print(f"성과 요약 CSV 저장: {NAV_2022_SUMMARY_FILE}")
display(strategy_vs_kospi200_nav_2022_summary.round(4))

plt.figure(figsize=(11, 5))
benchmark_plotted = False
for strategy, group in strategy_vs_kospi200_nav_2022.groupby("Strategy"):
    group = group.sort_values("Date").set_index("Date")
    group["StrategyNAV"].plot(label=f"{strategy} NAV")
    if not benchmark_plotted:
        group["BenchmarkNAV"].plot(
            label=f"{benchmark_name} NAV",
            linestyle="--",
            color="black",
        )
        benchmark_plotted = True
plt.title(f"2022-present NAV: PAS strategies vs {benchmark_name}")
plt.ylabel("NAV (common start = 1.0)")
plt.xlabel("Date")
plt.legend(); plt.grid(alpha=.25); plt.tight_layout(); plt.show()


## Export 영향이 큰 달의 Long/Short 선정 종목

Export가 바꿔 넣은 종목의 실현 롱숏 기여도가 가장 컸던 양(+)의 달 최대 3개와 해당 종목을 보여줍니다. 직전 12개월 HAC 유의성도 함께 표시합니다.


### 코드 설명 — 별도 모듈로 수출 신호가 종목 선정에 미친 영향을 점검한다.


In [ ]:
from export_month_selection_audit import run_export_month_selection_audit
export_monthly_audit, top_export_months, top_export_month_stocks = run_export_month_selection_audit(globals())
